In [ ]:
# Configure these widgets with real paths before running. Prefer a Unity Catalog Volume.
dbutils.widgets.text("input_csv", "<INPUT_CSV_PATH>")
dbutils.widgets.text("output_dir", "<OUTPUT_DIRECTORY>")
input_csv = dbutils.widgets.get("input_csv")
output_dir = dbutils.widgets.get("output_dir")
if any("<" in value or not value.strip() for value in (input_csv, output_dir)):
    raise ValueError("Set input_csv and output_dir widgets; see README.md")
if output_dir.rstrip("/") == input_csv.rstrip("/") or input_csv.startswith(output_dir.rstrip("/") + "/"):
    raise ValueError("Output must not contain or replace the input data")


In [ ]:
# Read once with the explicit schema in the next code cell.


## Explicit schema

Rank is an integer. Sales use double precision for aggregation; round only for display. Year remains a string because the source may contain missing-year markers.


In [ ]:
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
new_schema = StructType([
    StructField("Rank", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("Platform", StringType(), True),
    StructField("Year", StringType(), True),
    StructField("Genre", StringType(), True),
    StructField("Publisher", StringType(), True),
    StructField("North_American_Sales", DoubleType(), True),
    StructField("European_Sales", DoubleType(), True),
    StructField("Japanese_Sales", DoubleType(), True),
    StructField("Other_Sales", DoubleType(), True),
    StructField("Global_Sales", DoubleType(), True),

])
action = spark.read.option('mode', 'FAILFAST').csv(input_csv, schema=new_schema, header=True)
action.show()


## Normalize platform labels

Rename the Atari platform and preserve the source Rank. A Spark partition identifier is not a contiguous ranking.


In [ ]:
from pyspark.sql.functions import when

# Preserve the ranking supplied by the dataset; Spark partition IDs are not ranks.
action = action.withColumn("Platform", when(action["Platform"] == "2600", "Atari 2600").otherwise(action["Platform"]))
action.orderBy("Rank").show()


In [ ]:
# Refuse to overwrite prior results; choose a new output directory for each run.
action.write.mode("errorifexists").option("header", "true").csv(output_dir)


##Visualisations##

###Popularity by genre###
Now that we have cleaned the data and the games are already ranked by popularity, the count per each genre give the which genre contains the most number of popular games

In [ ]:
import plotly.express as px 
action.createOrReplaceGlobalTempView("action")
action.createOrReplaceTempView("workingdf") 

CountByGenre = spark.sql("SELECT Global_Sales, Name, Genre FROM workingdf") 
pandas_df = CountByGenre.toPandas() 

grouped_df = pandas_df.groupby("Genre").size().to_frame(name="count").reset_index() 
fig = px.bar(grouped_df, x="Genre", y="count", text="count")
fig.update_traces(texttemplate='%{text:.2s}', textposition='outside')
fig.show()


###Sales by Platform###
We use Spark to select global sales figure of each game and their plaform and then use Pandas to add the sales per each platform and present it in a pie chart. 

In [ ]:
import plotly.express as px 
action.createOrReplaceGlobalTempView("action")
action.createOrReplaceTempView("workingdf") 

SalesbyPlatform = spark.sql("SELECT Global_Sales, Platform FROM workingdf")  
grouped_df = SalesbyPlatform.toPandas().groupby("Platform")["Global_Sales"].sum().reset_index() 
fig = px.pie(grouped_df, values='Global_Sales', names='Platform', title='Distribution of Global Sales by Platform')
fig.show()

### Top ten source ranks

Sort explicitly by the original numeric rank before selecting ten rows.


In [ ]:
import plotly.express as px 
action.createOrReplaceGlobalTempView("action")
action.createOrReplaceTempView("workingdf") 

Rank = spark.sql("SELECT Rank, Name FROM workingdf ORDER BY Rank ASC, Name ASC LIMIT 10 ")
Rank.show()

###Further Insight###
##Top 10 game publishers by Global Sales##
We get the sum of global sales as total sales, and group total sales by publisher and take the top 10 

In [ ]:
import plotly.express as px 
action.createOrReplaceGlobalTempView("action")
action.createOrReplaceTempView("workingdf") 
sales_by_publisher = spark.sql("""SELECT Publisher, SUM(Global_Sales) as Total_Sales FROM workingdf GROUP BY Publisher ORDER BY Total_Sales DESC LIMIT 10""")
sales_by_publisher.show()
grouped_df = sales_by_publisher.toPandas().reset_index() 

fig = px.bar(grouped_df, x="Publisher", y="Total_Sales", title='Top 10 Game Publishers by Global Sales')
fig.show()

##Global sales by Franchise##
Here we are selecting games by four selected franchises using a text matcher and building a dataframe with the column "Franchise" to get a visualisation of sales of each franchise, and also get a count of games in each franchise to put it in the visualisation for comparison.

In [ ]:
import plotly.express as px
from pyspark.sql.functions import col, count, sum, when

# Create a temporary view for the action DataFrame
action.createOrReplaceGlobalTempView("action")

# Create the workingdf DataFrame by selecting all columns from the global_temp.action table
workingdf = spark.sql("SELECT * FROM global_temp.action")

# Apply the transformations on the workingdf DataFrame
df_transformed = workingdf.select(
    when(col("Name").like("Call of Duty%"), "Call of Duty")
    .when(col("Name").like("Grand Theft Auto%"), "Grand Theft Auto")
    .when(col("Name").like("Pokemon%"), "Pokemon")
    .when(col("Name").like("FIFA%"), "FIFA")
    .otherwise("Other").alias("Franchise"),
    col("Global_Sales")
)

# Perform aggregation on the transformed DataFrame
df_aggregated = df_transformed.filter(col("Franchise") != "Other") \
    .groupBy("Franchise") \
    .agg(count("*").alias("Game_Count"), sum("Global_Sales").alias("Total_Franchise_Sales")) \
    .orderBy("Total_Franchise_Sales", ascending=False)

# Convert the Spark DataFrame to a Pandas DataFrame for plotting
df_pandas = df_aggregated.toPandas()

# Create the Plotly visualization
fig = px.bar(df_pandas, x='Franchise', y=['Total_Franchise_Sales', 'Game_Count'], title='Total Franchise Sales')
fig.show()

##Average sales of publishers in different regions##
In this visualisation we explore the average sales of each publisher in 2016. This aggregates the game sales of the games of each publisher by first taking the average of the sales in different regions.

In [ ]:
import plotly.express as px
from pyspark.sql.functions import col, count, sum, when, avg, round

# Create a temporary view for the action DataFrame
action.createOrReplaceGlobalTempView("action")

# Create the workingdf DataFrame by selecting all columns from the global_temp.action table
workingdf = spark.sql("SELECT * FROM global_temp.action")

# Transform and aggregate the data
regional_sales = workingdf.groupBy("Name", "Year", "Publisher") \
    .agg(
        sum("North_American_Sales").alias("NA_Sales"),
        sum("European_Sales").alias("EU_Sales"),
        sum("Japanese_Sales").alias("JP_Sales"),
        sum("Other_Sales").alias("Other_Sales")
    )

# Aggregate the data again
df_aggregated = regional_sales.groupBy("Publisher", "Year") \
    .agg(
        round(avg("NA_Sales"), 2).alias("Avg_NA_Sales"),
        round(avg("EU_Sales"), 2).alias("Avg_EU_Sales"),
        round(avg("JP_Sales"), 2).alias("Avg_JP_Sales"),
        round(avg("Other_Sales"), 2).alias("Avg_Other_Sales")
    ) \
    .orderBy("Year", ascending=False)

# Filter the DataFrame for the year 2016
df_2016 = df_aggregated.filter(col("Year") == 2016)

# Convert the Spark DataFrame to a Pandas DataFrame
df_pandas = df_2016.toPandas()

# Calculate the total average sales for each publisher
df_pandas['Total_Avg_Sales'] = df_pandas[['Avg_NA_Sales', 'Avg_EU_Sales', 'Avg_JP_Sales', 'Avg_Other_Sales']].sum(axis=1)

# Sort the DataFrame in ascending order of total average sales
df_pandas = df_pandas.sort_values('Total_Avg_Sales')

# Create a bar chart with Plotly
fig = px.bar(df_pandas, x='Publisher', y='Total_Avg_Sales', title='Average Sales of Publishers in 2016',
             color='Publisher', text='Total_Avg_Sales')

# Update layout to display text on top of the bars
fig.update_traces(texttemplate='%{text:.2s}', textposition='outside')

fig.show()


#